# Head-direction decoders, Skromne Carrasco, Viejo & Peyrache (2026), dandiset 001676

Paper: doi:10.1038/s41586-025-10096-w. Claims: `claims/001676/10.1038-s41586-025-10096-w.yaml`.
Dataset: DANDI:001676, version 0.251205.2137 (the version the paper cites, and the only published
one). Files are streamed; nothing is written to disk. The other claims, the data check and the
reasons for the readings used here are in `reproduce_skromne_carrasco_2026.ipynb`.

| Claim | Status |
|---|---|
| edf8-same-vs-mixed | tested here |
| fig4e-decoder-single-visit | tested here |
| fig1k-decoding-across-days | blocked, too_large: about 1,900 decoder fits; the estimate from the fits timed here is printed at the end |

Decoder (Methods, Decoding using extreme gradient-boosted trees): HD cells with CellReg score
> 0.8, split-half correlation >= 0.4 and HDI >= 0.11 in the training session, kept when they meet
the same rule in the test session ("only the HD cells common to the testing day", Fig. 1j legend);
dF/F smoothed with a Gaussian of sigma 133 ms, divided by its maximum, averaged in 0.09 s bins; in
each session the bins with the top 50% of summed activity, z-scored; head direction in 32 bins;
XGBoost multiclass with the library defaults. Error: minimum circular difference between the
centre of the predicted bin and the head direction; chance: the same predictions against the
time-reversed head direction. Main reading of the HD rule as in the main notebook (12 degree
kernel, HDI on the smoothed curve).

Variants. (i) Raw HDI, as in the main notebook. (ii) The test-session rule loosened: cells meeting
the rule in the training session and registered by CellReg in the test session, whether or not
they meet it there; the Methods give the rule for the training session and say only "HD cells in
common between each training and testing session". Fits run four at a time, one thread each.

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy import stats
import xgboost as xgb
import hdstab as H

T0 = time.time()
pd.set_option("display.width", 160)
DEG = np.degrees
print("dandiset", H.DANDISET_ID, "version", H.DANDISET_VERSION, "xgboost", xgb.__version__)

In [ ]:
ASSETS = H.list_assets()
IMAGING = [a for a in ASSETS if "ophys" in a["path"]]
ALL = H.load_sessions(IMAGING, decoding=True, protocols=("multi", "single"), workers=12)
S = [s for s in ALL if not s.get("skipped")]
print("kept", len(S), "sessions of the multi-environment and single-visit protocols in %.0f s" % (time.time() - T0))

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ["mouse", "protocol", "env", "k", "nroi"]} for s in S])
print(pd.crosstab(info.mouse, [info.protocol, info.env]))
nb = np.array([len(s["dec_hd"]) for s in S])
print("decoder bins of 0.09 s per session: %d to %d" % (nb.min(), nb.max()))
x = np.concatenate([s["dec_x"].ravel() for s in S[:20]])
print("normalized activity (first 20 sessions) %.3f to %.3f; head direction per bin %.3f to %.3f rad"
      % (x.min(), x.max(), min(s["dec_hd"].min() for s in S), max(s["dec_hd"].max() for s in S)))

In [ ]:
from concurrent.futures import ThreadPoolExecutor

MAIN, RAW = ("12deg", "smooth"), ("12deg", "raw")


def tables(protocol, cfg=MAIN, **kw):
    out = {}
    for m in range(1, 11):
        g = H.group(S, m, protocol)
        if g:
            out[m] = H.hd_table(g, sigma=cfg[0], hdi_kind=cfg[1], **kw)
    return out


def decoder_cells(T):
    """Per session, registered cells meeting the decoder rule (score, split-half, HDI)."""
    return T["passes"] & (T["hdi"] >= H.HDI_SESSION_MIN) & (T["score"] > 0.8)[:, None]


def prep(s, rois):
    x = s["dec_x"][:, rois]
    pop = x.sum(1)
    keep = pop >= np.median(pop)
    x = x[keep]
    x = (x - x.mean(0)) / np.where(x.std(0) > 0, x.std(0), 1)
    return x, s["dec_hd"][keep].astype(float)


def decode(T, i, j, rule="both"):
    """Train on session i, test on session j. rule "both": cells meeting the decoder rule in both
    sessions; "train": meeting it in i and registered in j (variant ii)."""
    ok = decoder_cells(T)
    cells = np.where(ok[:, i] & (ok[:, j] if rule == "both" else T["reg"][:, j]))[0]
    if len(cells) < 2:
        return None
    g = T["sessions"]
    Xtr, htr = prep(g[i], T["R"][cells, i])
    Xte, hte = prep(g[j], T["R"][cells, j])
    ytr = np.minimum((htr / (2 * np.pi) * H.DEC_NCLASS).astype(int), H.DEC_NCLASS - 1)
    present = np.unique(ytr)
    remap = {c: k for k, c in enumerate(present)}
    clf = xgb.XGBClassifier(n_jobs=1, random_state=0)
    clf.fit(Xtr, np.array([remap[c] for c in ytr]))
    pred = present[clf.predict(Xte)]
    centre = (pred + 0.5) * 2 * np.pi / H.DEC_NCLASS
    err = H.wrap(centre - hte)
    err_rev = H.wrap(centre - hte[::-1])
    return dict(cells=len(cells), median_err=DEG(np.median(np.abs(err))), median_chance=DEG(np.median(np.abs(err_rev))),
                mean_err=DEG(np.mean(np.abs(err))), mean_chance=DEG(np.mean(np.abs(err_rev))),
                circ_var=1 - H.resultant(err), lin_var=np.var(err), bins_train=len(ytr), bins_test=len(hte))


def decode_all(tasks):
    """tasks: (table, i, j, rule); results in the same order, four fits at a time."""
    with ThreadPoolExecutor(4) as ex:
        return list(ex.map(lambda t: decode(*t), tasks))


VARIANTS = [("main (12deg/smooth, rule in both sessions)", MAIN, "both"),
            ("(i) 12deg/raw, rule in both sessions", RAW, "both"),
            ("(ii) 12deg/smooth, rule in training, registered in test", MAIN, "train")]
N_FITS, FIT_TIME = 0, 0.0

## fig4e-decoder-single-visit

> A decoder of head direction trained on the first and tested on the last session was accurate
> for both environments, with no difference between the weekly and single groups (weekly 35.9,
> single 43.9 degrees median error; P = 0.438).

In [ ]:
SINGLE = {cfg: tables("single", cfg, min_sessions=2) for cfg in [MAIN, RAW]}
F4 = {}
for label, cfg, rule in VARIANTS:
    meta, tasks = [], []
    for m in [6, 7, 8, 9, 10]:
        T = SINGLE[cfg][m]
        g = T["sessions"]
        for e in sorted(set(s["env"] for s in g)):
            js = [j for j, s in enumerate(g) if s["env"] == e]
            meta.append(dict(mouse=m, env=e, group="weekly" if len(js) > 2 else "single"))
            tasks.append((T, js[0], js[-1], rule))
    t_start = time.time()
    res = decode_all(tasks)
    N_FITS += len(tasks); FIT_TIME += time.time() - t_start
    f4 = pd.DataFrame([dict(**mt, **(r or {"cells": 0})) for mt, r in zip(meta, res)])
    F4[label] = f4
    print(label)
    print(f4.round(2).to_string(index=False))
    for grp, x, t in [("weekly", 35.9, 15.0), ("single", 43.9, 17.1)]:
        d = f4[f4.group == grp]
        print("   %-6s mean median error %.1f deg (s.d. %.1f; expected %.1f +- %.1f); below chance in %d of %d"
              % (grp, d.median_err.mean(), d.median_err.std(ddof=0), x, t, (d.median_err < d.median_chance).sum(), d.median_err.notna().sum()))
    wk = f4[f4.group == "weekly"].sort_values("mouse").median_err.values
    sg = f4[f4.group == "single"].sort_values("mouse").median_err.values
    ok = np.isfinite(wk) & np.isfinite(sg)
    w = stats.wilcoxon(wk[ok], sg[ok])
    print("   weekly vs single, two-sided Wilcoxon over %d mice: W %.1f p %.3f" % (ok.sum(), w.statistic, w.pvalue))

**Result: reproduced.** The mean median error is 42.5 degrees for weekly (within 35.9 +- 15.0) and 48.8 for single (within 43.9 +- 17.1). Every decoder is below its time-reversed chance (83 to 97 degrees), and the group test is not significant (W = 3, p = 0.312). Raw HDI (41.7 and 48.1; W = 4, p = 0.438) and the loosened test-session rule (44.0 and 49.4; W = 4, p = 0.438) give the same outcome, with every decoder below chance.

## edf8-same-vs-mixed

> For recordings 7 days apart, decoders trained and tested in the same environment tended to be
> better than across environments, and the change in pairwise offset tended to be smaller,
> neither significantly (p = 0.063 for both).

Pairs of sessions of the multi-environment protocol 7 days apart, trained on the earlier and
tested on the later; "single" when both are in one environment, "mixed" otherwise. Per mouse the
mean over pairs. The offset change per pair is the median over cell pairs of the unsigned offset
change (main reading of the main notebook) of HD cells valid in both (Fig. 2 rule, registered in
>= 6 sessions).

In [ ]:
MULTI = {cfg: tables("multi", cfg) for cfg in [MAIN, RAW]}
OFFS = {(cfg, m): H.all_offsets(MULTI[cfg][m]) for cfg in [MAIN, RAW] for m in [1, 2, 3, 4, 5]}
E8 = {}
for label, cfg, rule in VARIANTS:
    meta, tasks = [], []
    for m in [1, 2, 3, 4, 5]:
        T = MULTI[cfg][m]
        g = T["sessions"]
        dd = H.day_diff(g)
        for i in range(len(g)):
            for j in range(i + 1, len(g)):
                if dd[i, j] != 7:
                    continue
                per, _ = H.offset_changes(OFFS[(cfg, m)], [(i, j)], unsigned=True)
                meta.append(dict(mouse=m, i=i, j=j, kind="single" if g[i]["env"] == g[j]["env"] else "mixed",
                                 offset_change=np.median(per[0]) if len(per[0]) else np.nan))
                tasks.append((T, i, j, rule))
    t_start = time.time()
    res = decode_all(tasks)
    N_FITS += len(tasks); FIT_TIME += time.time() - t_start
    e8 = pd.DataFrame([dict(**mt, **(r or {"cells": 0})) for mt, r in zip(meta, res)])
    E8[label] = e8
    print(label)
    print("   pairs 7 days apart: %d, fits %.0f s" % (len(e8), time.time() - t_start))
    print(e8.groupby(["mouse", "kind"]).size().unstack())
    per_mouse = e8.groupby(["mouse", "kind"])[["offset_change", "circ_var", "lin_var", "median_err", "median_chance"]].mean().unstack()
    print(per_mouse.round(3))
    for col, exp in [("offset_change", (16.2, 10.1, 23.1, 11.4)), ("circ_var", (0.58, 0.09, 0.74, 0.13)), ("lin_var", (0.58, 0.09, 0.74, 0.13))]:
        s_, m_ = per_mouse[(col, "single")].values, per_mouse[(col, "mixed")].values
        ok = np.isfinite(s_) & np.isfinite(m_)
        w = stats.wilcoxon(s_[ok], m_[ok])
        print("   %-13s single %.3f (s.d. %.3f) vs mixed %.3f (s.d. %.3f); single lower in %d of %d mice; two-sided Wilcoxon W %.1f p %.3f; expected %.2f +- %.2f vs %.2f +- %.2f"
              % (col, s_[ok].mean(), s_[ok].std(), m_[ok].mean(), m_[ok].std(), (s_[ok] < m_[ok]).sum(), ok.sum(), w.statistic, w.pvalue, *exp))
    print("   decoders below chance (median error): %d of %d pairs; dropped (fewer than 2 cells): %d"
          % ((e8.median_err < e8.median_chance).sum(), e8.median_err.notna().sum(), e8.median_err.isna().sum()))

**Result: partially reproduced.** Under the main reading, single-environment pairs are lower than mixed pairs in 4 of 5 mice for the offset change (15.2 against 19.5 degrees, within the paper's means; p = 0.125) and for the circular variance of the decoder error (0.53 against 0.65, within tolerance; p = 0.125). They are lower in 5 of 5 for the linear variance (W = 0, p = 0.062, as in the paper), but its values (1.91 against 3.15) are far from the paper's 0.58 and 0.74. With raw HDI the offset change matches fully (19.6 against 27.1 degrees, within tolerance, lower in 5 of 5, W = 0, p = 0.062), but the decoder still does not: circular variance 0.51 against 0.64, lower in 4 of 5 (p = 0.125); linear variance lower in 5 of 5 with values far from the paper's. The loosened test-session rule gives the same as the main reading. No reading gives W = 0 with means within tolerance for both measures.

## fig1k-decoding-across-days (blocked, too_large)

The claim's decoder runs on every ordered pair of sessions of the circular protocol, and on each
session split in halves ("same"). The estimate below uses the time per fit measured above, over
all the fits of this notebook run four at a time (the edf8 and fig4e decoders, on the same cell
rule), and also gives the part of the claim's partial reading (pairs up to 10 days apart).

In [ ]:
circ = pd.DataFrame([dict(mouse=s["mouse"], k=s["k"], date=s["start"][:10]) for s in ALL if s["protocol"] == "circular"])
n_per_mouse = circ.groupby("mouse").size()
n_fits = int((n_per_mouse * (n_per_mouse - 1)).sum() + n_per_mouse.sum())
n_10 = n_per_mouse.sum()
for m, g in circ.groupby("mouse"):
    d = pd.to_datetime(g.date).values
    dd = np.abs((d[:, None] - d[None, :]) / np.timedelta64(1, "D"))
    n_10 += int(((dd <= 10) & (dd > 0)).sum())
per_fit = FIT_TIME / N_FITS
print("circular sessions per mouse:", n_per_mouse.to_dict())
print("ordered pairs plus same-session fits: %d (%d up to 10 days apart); %.2f s per fit measured above over %d fits, four at a time"
      % (n_fits, n_10, per_fit, N_FITS))
print("estimate %.0f min for all pairs and %.0f min up to 10 days, against the 60 min limit of a notebook, plus loading the circular sessions"
      % (n_fits * per_fit / 60, n_10 * per_fit / 60))
print("total %.0f s" % (time.time() - T0))

## Summary

| Claim | Outcome |
|---|---|
| fig4e-decoder-single-visit | reproduced (all three variants) |
| edf8-same-vs-mixed | partially reproduced (offset change matches with raw HDI; the decoder does not under any variant) |
| fig1k-decoding-across-days | blocked, too_large (estimate printed above) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

The decoders follow the Methods. XGBoost's library defaults are used because the paper gives no
hyperparameters. Two choices are assumptions: normalization by each cell's maximum, and the
cell rule (HD cells meeting the rule in both sessions); the loosened rule (registered in the test
session) and raw HDI are shown and give the same outcomes. The linear variance of the signed
error (radians squared) is far from the paper's values, while the circular variance is within
tolerance. Fig. 1k needs 1,883 fits, estimated at 80 minutes at the rate measured here (51 minutes
for the pairs up to 10 days apart of its partial reading), plus loading the 97 circular sessions
(the 227 sessions above took 1125 s), over the 60 minute limit of a notebook. Findings about the dataset are in the main notebook.